In [1]:
"""
incident_flag 피처 생성

data/국토교통부_사고돌발정보.csv(전국 돌발상황 이력, 95,054건)를
우리 706개 타겟 링크(segment_link_mapping.csv 기준)로 필터링하고,
10분 단위(time_slot)로 그 구간x방향에 활성 돌발상황이 있었는지 플래그화한다.

이 피처는 우리 속도 시계열과 무관한 외부 이벤트 데이터라(lane_remain_ratio와
동일 카테고리) Train 기간 컷오프에 따라 값이 바뀌지 않는다. 즉 leakage
걱정 없이 전체 기간에 대해 한 번만 계산해서 재사용하면 된다.

grain을 10분으로 잡은 이유: Prophet 산출물(prophet/prophet/*.csv)이 이미
10분 단위이고, 원래 파이프라인 설계도 10분 단위 row를 가정하므로 맞춘다.
돌발상황은 지속시간이 짧아(중앙값 30분) 일 단위로 만들면 신호가 거의 다
사라지므로 반드시 세밀한 grain이 필요하다.

종료시각 결측(전체의 약 52%, '-') 처리:
  종료시각이 있는 건들의 지속시간 중앙값이 정확히 30분이므로, 결측 시
  시작시각 + 30분을 기본 종료시각으로 채운다. 단 원본에 며칠~수백일짜리
  이상치(데이터 오류로 추정)가 섞여있어, 모든 이벤트의 지속시간을 24시간으로
  상한(clip)한다 - 실제 돌발상황이 그렇게 오래 같은 상태로 지속되며 교통에
  영향을 주는 경우는 드물다고 보는 게 합리적인 가정.

출력: output/features/incident_flag_10min.parquet
  segment_key, time_slot(HHMM 정수, 10분 단위), incident_flag(bool),
  incident_count(그 슬롯에 겹친 돌발 건수)
"""

from pathlib import Path

import pandas as pd
import polars as pl

INCIDENT_PATH = "./data/국토교통부_사고돌발정보.csv"
SEGMENT_LINK_MAPPING_PATH = "./output/segment_link_mapping.csv"
SPEED_PATH = "./output/segment_weighted_speed_final.parquet"

DEFAULT_DURATION_MIN = 30  # 종료시각 결측 시 기본 지속시간(중앙값 기반)
MAX_DURATION_MIN = 24 * 60  # 이상치 방지용 상한(24시간)

OUTPUT_DIR = Path("./output/features")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

In [2]:
# ==================================================================
# 1. 원본 로드 + 정리
# ==================================================================

raw = pd.read_csv(INCIDENT_PATH, encoding="utf-8")
raw.columns = [
    "start_dt", "agency", "road_type", "road_no", "road_name", "direction_text",
    "category", "sub_category", "detail", "X", "Y", "link_id", "end_dt",
]

raw["start_dt"] = pd.to_datetime(raw["start_dt"], errors="coerce")
raw["end_dt_raw"] = pd.to_datetime(raw["end_dt"].replace("-", pd.NA), errors="coerce")

# 종료시각 결측 -> 시작시각 + 기본 지속시간
raw["end_dt"] = raw["end_dt_raw"].fillna(raw["start_dt"] + pd.Timedelta(minutes=DEFAULT_DURATION_MIN))

# 지속시간 상한 적용(이상치 방지) + 시작>종료(데이터 오류) 행 제거
max_end = raw["start_dt"] + pd.Timedelta(minutes=MAX_DURATION_MIN)
raw["end_dt"] = raw[["end_dt"]].join(max_end.rename("max_end")).min(axis=1)
raw = raw[raw["end_dt"] >= raw["start_dt"]].copy()

print(f"원본 행 수: {len(raw)}")
print(f"종료시각 결측(기본값 적용) 비율: {raw['end_dt_raw'].isna().mean():.1%}")
print(f"기간: {raw['start_dt'].min()} ~ {raw['start_dt'].max()}")
print(f"category 분포:\n{raw['category'].value_counts()}")

원본 행 수: 95049
종료시각 결측(기본값 적용) 비율: 52.1%
기간: 2023-01-01 08:45:54 ~ 2026-07-10 22:47:34
category 분포:
category
공사      39984
기타돌발    27853
교통사고    16555
기상       5973
기타       3258
재난       1426
Name: count, dtype: int64


In [3]:
# ==================================================================
# 2. 우리 타겟 링크로 필터링 + segment_key 매핑
# ==================================================================

seg_map = pd.read_csv(SEGMENT_LINK_MAPPING_PATH, encoding="utf-8")
seg_map["segment_key"] = seg_map["segment_id"] + "_" + seg_map["direction"]
link_to_segments = seg_map[["link_id", "segment_key"]].drop_duplicates()

raw["link_id"] = pd.to_numeric(raw["link_id"], errors="coerce")

matched = raw.merge(link_to_segments, on="link_id", how="inner")

print(f"타겟 링크와 매칭되는 돌발 건수: {len(matched)} / {len(raw)}")
print(f"매칭된 segment_key 수: {matched['segment_key'].nunique()} / {seg_map['segment_key'].nunique()}")
print()
print(matched["category"].value_counts())

타겟 링크와 매칭되는 돌발 건수: 4236 / 95049
매칭된 segment_key 수: 88 / 90

category
공사      2893
교통사고     983
기타       353
기상         7
Name: count, dtype: int64


In [4]:
# ==================================================================
# 3. 10분 단위(segment_key, time_slot) 플래그 그리드 생성
# ==================================================================
# 각 돌발 이벤트를 [start_dt, end_dt] 범위의 10분 슬롯으로 explode한 뒤,
# 같은 슬롯에 여러 건이 겹치면 incident_flag=True, incident_count=겹친 건수.

speed_df = pl.read_parquet(SPEED_PATH, columns=["timestamp"])
ts_min = speed_df["timestamp"].min()
ts_max = speed_df["timestamp"].max()
print(f"타임스탬프 범위(속도 데이터 기준): {ts_min} ~ {ts_max}")

# 학습 기간과 겹치는 이벤트만 남겨서 explode 비용을 줄임
matched = matched[(matched["end_dt"] >= ts_min) & (matched["start_dt"] <= ts_max)].copy()
matched["start_dt"] = matched["start_dt"].clip(lower=ts_min)
matched["end_dt"] = matched["end_dt"].clip(upper=ts_max)
print(f"학습 기간과 겹치는 돌발 건수: {len(matched)}")


def floor_10min(ts: pd.Timestamp) -> pd.Timestamp:
    return ts.floor("10min")


expanded_rows = []
for row in matched.itertuples(index=False):
    slot_start = floor_10min(row.start_dt)
    slot_end = floor_10min(row.end_dt)
    slots = pd.date_range(slot_start, slot_end, freq="10min")
    expanded_rows.append(pd.DataFrame({"segment_key": row.segment_key, "timestamp": slots}))

expanded = pd.concat(expanded_rows, ignore_index=True)
flag_grid = (
    expanded.groupby(["segment_key", "timestamp"])
    .size()
    .reset_index(name="incident_count")
)
flag_grid["incident_flag"] = True

print(f"\nincident_flag=True 슬롯 수: {len(flag_grid)}")
print(flag_grid.head())

타임스탬프 범위(속도 데이터 기준): 2024-10-01 00:00:00 ~ 2026-07-01 23:55:00
학습 기간과 겹치는 돌발 건수: 1892

incident_flag=True 슬롯 수: 33764
         segment_key           timestamp  incident_count  incident_flag
0  SEG_01_201_202_AB 2025-03-10 08:10:00               1           True
1  SEG_01_201_202_AB 2025-03-10 08:20:00               1           True
2  SEG_01_201_202_AB 2025-03-10 08:30:00               1           True
3  SEG_01_201_202_AB 2025-03-10 08:40:00               1           True
4  SEG_01_201_202_AB 2025-05-02 11:20:00               1           True


In [5]:
# ==================================================================
# 4. 저장
# ==================================================================
# is_bottleneck_slot과 마찬가지로, 여기서는 "돌발이 있었던 슬롯"만 남기고
# 저장한다(sparse). 전체 그리드가 필요하면 피처 매트릭스 조립 시
# left join 후 incident_flag를 False/incident_count를 0으로 채우면 된다
# (아래 join_incident_flag 참고).

out = flag_grid[["segment_key", "timestamp", "incident_flag", "incident_count"]].sort_values(
    ["segment_key", "timestamp"]
)
pl.from_pandas(out).write_parquet(OUTPUT_DIR / "incident_flag_10min.parquet")

print(f"저장 완료: {(OUTPUT_DIR / 'incident_flag_10min.parquet').resolve()}")
print(f"shape: {out.shape}")


def join_incident_flag(feature_df: pl.DataFrame, incident_lookup: pl.DataFrame) -> pl.DataFrame:
    """
    feature_df: segment_key, timestamp(10분 단위)를 포함한 XGBoost 피처 매트릭스
    incident_lookup: 이 노트북에서 저장한 incident_flag_10min.parquet
    """
    return feature_df.join(
        incident_lookup, on=["segment_key", "timestamp"], how="left"
    ).with_columns(
        [
            pl.col("incident_flag").fill_null(False),
            pl.col("incident_count").fill_null(0),
        ]
    )

저장 완료: C:\Users\6152\Desktop\물류\output\features\incident_flag_10min.parquet
shape: (33764, 4)


In [6]:
# ==================================================================
# 5. 검증
# ==================================================================

n_segments = out["segment_key"].nunique()
total_slots = len(pl.read_parquet(SPEED_PATH, columns=["timestamp"])["timestamp"].unique())
print(f"돌발 플래그가 존재하는 segment_key 수: {n_segments} / {seg_map['segment_key'].nunique()}")
print(f"segment_key별 incident_flag 슬롯 수 상위 10개:")
print(out.groupby("segment_key").size().sort_values(ascending=False).head(10))
print()
print(f"incident_count 분포:")
print(out["incident_count"].value_counts().sort_index())

돌발 플래그가 존재하는 segment_key 수: 83 / 90
segment_key별 incident_flag 슬롯 수 상위 10개:
segment_key
SEG_15_215_216_AB    1954
SEG_03_203_204_AB    1891
SEG_35_235_236_AB    1740
SEG_01_201_202_AB    1640
SEG_38_238_239_BA    1564
SEG_44_243_244_BA    1478
SEG_15_215_216_BA    1388
SEG_03_203_204_BA    1376
SEG_38_238_239_AB    1263
SEG_37_237_238_AB    1178
dtype: int64

incident_count 분포:
incident_count
1    32186
2     1510
3       22
4       46
Name: count, dtype: int64
